In [1]:
import emcee
import corner
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import pickle
from uncertainties import ufloat
from uncertainties.umath import *
from uncertainties import unumpy as unp
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
cosmo=FlatLambdaCDM(70, 0.3)
from functions.statistics_functions import chi_squared
from functions.load_data_and_var import generate_nfws_from_combined_dataset_werr, load_data_and_var

In [3]:
reflex_data = pd.read_csv('data/reflex70.txt', header=7, sep='|', skiprows=[range(6), 8],on_bad_lines='warn', skipfooter=1, skipinitialspace=True).rename(columns=lambda x: x.strip())
reflex_data = reflex_data.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)[['RXC', 'Flux', 'e_Flux', 'ObsLum', 'LumCor', 'z']].dropna()

/var/folders/99/vfh56jrs1nqcj8zqtcs5fkv40000gn/T/ipykernel_94075/662693947.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  reflex_data = pd.read_csv('data/reflex70.txt', header=7, sep='|', skiprows=[range(6), 8],on_bad_lines='warn', skipfooter=1, skipinitialspace=True).rename(columns=lambda x: x.strip())


In [4]:
reflex_data

,RXC,Flux,e_Flux,ObsLum,LumCor,z
0,RXCJ0003.1-0605,4.497,12.4,6.395,6.876,0.2320
1,RXCJ0003.2-3555,7.537,17.8,0.421,0.478,0.0490
2,RXCJ0003.8+0203,4.155,18.8,0.855,0.929,0.0924
3,RXCJ0006.0-3443,5.832,13.6,1.875,1.995,0.1147
4,RXCJ0011.3-2851,12.014,9.0,1.089,1.197,0.0620
...,...,...,...,...,...,...
442,RXCJ2351.6-2605,9.295,10.1,12.422,13.651,0.2264
443,RXCJ2354.2-1024,9.622,12.1,1.337,1.469,0.0765
444,RXCJ2357.0-3445,32.663,5.5,1.698,1.787,0.0475
445,RXCJ2359.3-6042,4.544,20.5,1.080,1.149,0.0989


In [5]:
mcxccls=pd.read_csv('data/mcxc|.txt', header=3, sep='|', skiprows=[4],on_bad_lines='warn', skipfooter=1).rename(columns=lambda x: x.strip())
mcxccls = mcxccls.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)[['MCXC', 'OName', 'L500', 'Sub-Cat']].dropna()
mcxccls

/var/folders/99/vfh56jrs1nqcj8zqtcs5fkv40000gn/T/ipykernel_94075/3078319749.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  mcxccls=pd.read_csv('data/mcxc|.txt', header=3, sep='|', skiprows=[4],on_bad_lines='warn', skipfooter=1).rename(columns=lambda x: x.strip())


,MCXC,OName,L500,Sub-Cat
0,J0000.1+0816,RXC J0000.1+0816,0.196280,NORAS
1,J0000.4-0237,RXC J0000.4-0237,0.052338,SGP
2,J0001.6-1540,RXC J0001.6-1540,0.814902,SGP
3,J0001.9+1204,RXC J0001.9+1204,1.990205,NORAS
4,J0003.1-0605,RXCJ0003.1-0605,6.107094,REFLEX
...,...,...,...,...
1738,J2357.0-3445,RXCJ2357.0-3445,1.629622,REFLEX
1739,J2359.3-6042,RXCJ2359.3-6042,1.051539,REFLEX
1740,J2359.4-3418,MS2356.9-3434,0.223131,EMSS_1994
1741,J2359.5-3211,RX J2359.5-3211,1.069996,SHARC_SOUTH


In [6]:
reflex_mcxc_combo = pd.merge(mcxccls, reflex_data, left_on='OName', right_on='RXC',how='inner')
reflex_mcxc_combo

,MCXC,OName,L500,Sub-Cat,RXC,Flux,e_Flux,ObsLum,LumCor,z
0,J0003.1-0605,RXCJ0003.1-0605,6.107094,REFLEX,RXCJ0003.1-0605,4.497,12.4,6.395,6.876,0.2320
1,J0003.2-3555,RXCJ0003.2-3555,0.441862,REFLEX,RXCJ0003.2-3555,7.537,17.8,0.421,0.478,0.0490
2,J0003.8+0203,RXCJ0003.8+0203,0.846881,REFLEX,RXCJ0003.8+0203,4.155,18.8,0.855,0.929,0.0924
3,J0006.0-3443,RXCJ0006.0-3443,1.809494,REFLEX,RXCJ0006.0-3443,5.832,13.6,1.875,1.995,0.1147
4,J0011.3-2851,RXCJ0011.3-2851,1.086002,REFLEX,RXCJ0011.3-2851,12.014,9.0,1.089,1.197,0.0620
...,...,...,...,...,...,...,...,...,...,...
436,J2351.6-2605,RXCJ2351.6-2605,11.968015,REFLEX,RXCJ2351.6-2605,9.295,10.1,12.422,13.651,0.2264
437,J2354.2-1024,RXCJ2354.2-1024,1.336537,REFLEX,RXCJ2354.2-1024,9.622,12.1,1.337,1.469,0.0765
438,J2357.0-3445,RXCJ2357.0-3445,1.629622,REFLEX,RXCJ2357.0-3445,32.663,5.5,1.698,1.787,0.0475
439,J2359.3-6042,RXCJ2359.3-6042,1.051539,REFLEX,RXCJ2359.3-6042,4.544,20.5,1.080,1.149,0.0989


In [7]:
psz = pd.read_csv('data/psz2.txt', sep='|', header=9, skiprows=[10], on_bad_lines='warn', skipfooter=1)
psz = psz.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)#[['MCXC', 'MSZ', 'E_MSZ', 'e_MSZ', 'z']]#.dropna()
psz.columns=psz.columns.str.strip()
#psz.columns.tolist()
psz=psz[['z',
 'MSZ',
 'E_MSZ',
 'e_MSZ',
 'MCXC']].dropna()
psz

/var/folders/99/vfh56jrs1nqcj8zqtcs5fkv40000gn/T/ipykernel_94075/1052054181.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  psz = pd.read_csv('data/psz2.txt', sep='|', header=9, skiprows=[10], on_bad_lines='warn', skipfooter=1)


,z,MSZ,E_MSZ,e_MSZ,MCXC
0,0.1198,3.962411,0.393290,0.370242,J1516.5-0056
1,0.1710,5.122391,0.351061,0.322839,J1334.1+2013
2,0.1651,5.297053,0.320350,0.344108,J2104.3-4120
3,0.3416,6.333562,0.589880,0.611330,J2031.8-4037
4,-1.0000,0.000000,0.000000,0.000000,
...,...,...,...,...,...
1648,0.0957,2.523271,0.246912,0.250808,J2331.2-3630
1649,0.1786,5.224710,0.317408,0.302077,J2315.7-3746
1650,0.1496,4.721086,0.392048,0.389491,J2012.0-4129
1651,-1.0000,0.000000,0.000000,0.000000,


In [8]:
combo = pd.merge(reflex_mcxc_combo, psz, on="MCXC", how="inner")
combo

,MCXC,OName,L500,Sub-Cat,RXC,Flux,e_Flux,ObsLum,LumCor,z_x,z_y,MSZ,E_MSZ,e_MSZ
0,J0003.1-0605,RXCJ0003.1-0605,6.107094,REFLEX,RXCJ0003.1-0605,4.497,12.4,6.395,6.876,0.2320,0.2320,5.957353,0.368165,0.369144
1,J0003.8+0203,RXCJ0003.8+0203,0.846881,REFLEX,RXCJ0003.8+0203,4.155,18.8,0.855,0.929,0.0924,0.0924,2.238774,0.282160,0.301448
2,J0006.0-3443,RXCJ0006.0-3443,1.809494,REFLEX,RXCJ0006.0-3443,5.832,13.6,1.875,1.995,0.1147,0.1147,4.001657,0.248869,0.226974
3,J0011.3-2851,RXCJ0011.3-2851,1.086002,REFLEX,RXCJ0011.3-2851,12.014,9.0,1.089,1.197,0.0620,0.0620,2.606369,0.182667,0.188038
4,J0013.6-1930,RXCJ0013.6-1930,1.236332,REFLEX,RXCJ0013.6-1930,6.071,11.3,1.285,1.353,0.0940,0.0940,2.756623,0.233331,0.245311
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,J2347.7-2808,RXCJ2347.7-2808,1.029501,REFLEX,RXCJ2347.7-2808,49.578,3.8,1.014,1.127,0.0300,0.0300,1.480293,0.114748,0.107849
221,J2351.6-2605,RXCJ2351.6-2605,11.968015,REFLEX,RXCJ2351.6-2605,9.295,10.1,12.422,13.651,0.2264,0.2264,7.557539,0.365168,0.382711
222,J2354.2-1024,RXCJ2354.2-1024,1.336537,REFLEX,RXCJ2354.2-1024,9.622,12.1,1.337,1.469,0.0765,0.0765,2.761984,0.216913,0.225417
223,J2357.0-3445,RXCJ2357.0-3445,1.629622,REFLEX,RXCJ2357.0-3445,32.663,5.5,1.698,1.787,0.0475,0.0475,2.468719,0.163126,0.149009


In [9]:
combo['z_x']!=combo['z_y']

0      False
1      False
2      False
3      False
4      False
       ...  
220    False
221    False
222    False
223    False
224    False
Length: 225, dtype: bool

In [10]:
low_z_clusters = combo[combo['z_y']<0.2]

In [11]:
low_z_clusters

,MCXC,OName,L500,Sub-Cat,RXC,Flux,e_Flux,ObsLum,LumCor,z_x,z_y,MSZ,E_MSZ,e_MSZ
1,J0003.8+0203,RXCJ0003.8+0203,0.846881,REFLEX,RXCJ0003.8+0203,4.155,18.8,0.855,0.929,0.0924,0.0924,2.238774,0.282160,0.301448
2,J0006.0-3443,RXCJ0006.0-3443,1.809494,REFLEX,RXCJ0006.0-3443,5.832,13.6,1.875,1.995,0.1147,0.1147,4.001657,0.248869,0.226974
3,J0011.3-2851,RXCJ0011.3-2851,1.086002,REFLEX,RXCJ0011.3-2851,12.014,9.0,1.089,1.197,0.0620,0.0620,2.606369,0.182667,0.188038
4,J0013.6-1930,RXCJ0013.6-1930,1.236332,REFLEX,RXCJ0013.6-1930,6.071,11.3,1.285,1.353,0.0940,0.0940,2.756623,0.233331,0.245311
5,J0014.3-6604,RXCJ0014.3-6604,2.827248,REFLEX,RXCJ0014.3-6604,4.485,13.8,2.907,3.160,0.1599,0.1599,3.030337,0.328876,0.347930
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,J2344.2-0422,RXCJ2344.2-0422,1.852614,REFLEX,RXCJ2344.2-0422,12.638,7.4,1.855,2.061,0.0786,0.0786,3.116566,0.176734,0.208566
220,J2347.7-2808,RXCJ2347.7-2808,1.029501,REFLEX,RXCJ2347.7-2808,49.578,3.8,1.014,1.127,0.0300,0.0300,1.480293,0.114748,0.107849
222,J2354.2-1024,RXCJ2354.2-1024,1.336537,REFLEX,RXCJ2354.2-1024,9.622,12.1,1.337,1.469,0.0765,0.0765,2.761984,0.216913,0.225417
223,J2357.0-3445,RXCJ2357.0-3445,1.629622,REFLEX,RXCJ2357.0-3445,32.663,5.5,1.698,1.787,0.0475,0.0475,2.468719,0.163126,0.149009


In [12]:
import pickle
with open('data/reflex_mcxc_psz_low_z_clusters.p', 'wb') as fp:
    pickle.dump(low_z_clusters, fp)

In [13]:
data, var = load_data_and_var()

In [14]:
data

,MCXC,OName,L500,Sub-Cat,RXC,Flux,e_Flux,ObsLum,LumCor,z_x,z_y,MSZ,E_MSZ,e_MSZ,L_err
1,J0003.8+0203,RXCJ0003.8+0203,0.846881,REFLEX,RXCJ0003.8+0203,4.155,18.8,0.855,0.929,0.0924,0.0924,2.238774,0.282160,0.301448,0.136707
2,J0006.0-3443,RXCJ0006.0-3443,1.809494,REFLEX,RXCJ0006.0-3443,5.832,13.6,1.875,1.995,0.1147,0.1147,4.001657,0.248869,0.226974,0.211627
3,J0011.3-2851,RXCJ0011.3-2851,1.086002,REFLEX,RXCJ0011.3-2851,12.014,9.0,1.089,1.197,0.0620,0.0620,2.606369,0.182667,0.188038,0.086428
4,J0013.6-1930,RXCJ0013.6-1930,1.236332,REFLEX,RXCJ0013.6-1930,6.071,11.3,1.285,1.353,0.0940,0.0940,2.756623,0.233331,0.245311,0.124160
5,J0014.3-6604,RXCJ0014.3-6604,2.827248,REFLEX,RXCJ0014.3-6604,4.485,13.8,2.907,3.160,0.1599,0.1599,3.030337,0.328876,0.347930,0.313974
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,J2344.2-0422,RXCJ2344.2-0422,1.852614,REFLEX,RXCJ2344.2-0422,12.638,7.4,1.855,2.061,0.0786,0.0786,3.116566,0.176734,0.208566,0.119209
220,J2347.7-2808,RXCJ2347.7-2808,1.029501,REFLEX,RXCJ2347.7-2808,49.578,3.8,1.014,1.127,0.0300,0.0300,1.480293,0.114748,0.107849,0.035784
222,J2354.2-1024,RXCJ2354.2-1024,1.336537,REFLEX,RXCJ2354.2-1024,9.622,12.1,1.337,1.469,0.0765,0.0765,2.761984,0.216913,0.225417,0.140721
223,J2357.0-3445,RXCJ2357.0-3445,1.629622,REFLEX,RXCJ2357.0-3445,32.663,5.5,1.698,1.787,0.0475,0.0475,2.468719,0.163126,0.149009,0.084855


In [15]:
var

<Quantity [0.13670661, 0.21162745, 0.08642763, 0.12415974, 0.31397369,
           0.2846928 , 0.04313728, 0.30714761, 0.16384222, 0.09262341,
           0.15257523, 0.22813716, 0.07297288, 0.08604661, 0.03988705,
           0.01343183, 0.04116458, 0.05526068, 0.26658098, 0.41251589,
           0.24935543, 0.06529352, 0.08461261, 0.14423532, 0.08368238,
           0.13354662, 0.05293967, 0.09600298, 0.68521146, 0.51106161,
           0.16840712, 0.08693578, 0.06420011, 0.08803048, 0.05397613,
           0.05095171, 0.06468251, 0.05974175, 0.09247889, 0.08103832,
           0.27186057, 0.04420151, 0.2253449 , 0.12611217, 0.11223306,
           0.30265892, 0.72202926, 0.37494298, 0.34856102, 0.08215428,
           0.02087464, 0.1348663 , 0.28061397, 0.03238636, 0.14025306,
           0.04511578, 0.22616229, 0.23121924, 0.12950357, 0.04108408,
           0.04055649, 0.32180458, 0.04215004, 0.44792852, 0.12270911,
           0.11068043, 0.11532253, 0.23935962, 0.46274367, 0.35807988,
      

In [16]:
all_nfws = generate_nfws_from_combined_dataset_werr(data, var)

In [17]:
nfws = [nfw for nfw in all_nfws if nfw.energies.is_virialized_gaspari]

In [18]:
len(nfws)

111